In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.window import Window

import pyspark.sql.functions as F
import pandas as pd
import os

# spark = SparkSession.builder.master("local")\
#         .config("spark.port.maxRetries", 100)\
#         .config("spark.executor.instances", "6")\
#         .config("spark.executor.cores", "4")\
#         .config("spark.executor.memory", "8G")\
#         .config("spark.driver.memory", "2G")\
#         .config("spark.dynamicAllocation.enabled", "false")\
#         .config("spark.yarn.queue", "Low")\
#         .config("spark.port.maxRetries", 100)\
#         .appName("New_Data_Reader_V0")\
#         .getOrCreate()

spark = SparkSession.builder.master("local").getOrCreate()

In [5]:
import os
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.functions import lit

cin_station_num = "72429793812"
flo_station_num = "99495199999"
stations = [
    (cin_station_num, "Cincinnati"),
    (flo_station_num, "Florida")
]
years    = list(range(2015, 2025))
base_dir = r"C:\Users\namng\OneDrive\Documents\CloudComputingProjects\project4-pyspark\Data"

# 1) Read & tag each station-year CSV exactly once
dfs = []
for station_code, loc in stations:
    for yr in years:
        path = os.path.join(base_dir, str(yr), f"{station_code}.csv")
        if os.path.exists(path):
            df_temp = (
                spark.read
                     .option("header",      "true")
                     .option("inferSchema", "true")
                     .option("delimiter",   ",")
                     .csv(path)
                     # tag columns so you know station/year later
                     .withColumn("year",    lit(yr))
                     .withColumn("station", lit(station_code))
            )
            dfs.append(df_temp)
        else:
            print(f"⚠️  Missing {station_code} for {yr}, skipping")

# 2) Union them all into one DataFrame
df_all = reduce(DataFrame.unionByName, dfs)

# 3) Cache it in memory and trigger the load
df_all = df_all.cache()
print("Total records across all station-years:", df_all.count())

# Now df_all is “hot” in memory — all your subsequent operations will be fast.

# Example: show the distinct years to verify you have 10
df_all.select("year").distinct().orderBy("year").show()

# From here on, use df_all instead of re-reading files.


⚠️  Missing 99495199999 for 2016, skipping
Total records across all station-years: 6136
+----+
|year|
+----+
|2015|
|2016|
|2017|
|2018|
|2019|
|2020|
|2021|
|2022|
|2023|
|2024|
+----+



In [13]:
from pyspark.sql.functions import to_date, col, row_number, desc, asc
from pyspark.sql.window import Window

# 1) Convert DATE strings to real dates (and reuse the existing 'year' column)
df2 = (
    df_all
      .withColumn("DATE", to_date(col("DATE"), "yyyy-MM-dd"))
).filter(col("MAX") != 9999.9)

# 2) Define a window: one partition per year, order by MAX desc then DATE asc
w = Window.partitionBy("year") \
          .orderBy(desc(col("MAX")), asc(col("DATE")))

# 3) Pick exactly one row per year (the highest MAX, earliest date on ties)
hottest_per_year = (
    df2
      .withColumn("rn", row_number().over(w))
      .filter(col("rn") == 1)
      .select("year", "STATION", "NAME", "DATE", "MAX")
      .orderBy("year")
)

# 4) Show the 10 results
print("Hottest day per year (STATION, NAME, DATE, MAX):")
hottest_per_year.show(10, truncate=50)


Hottest day per year (STATION, NAME, DATE, MAX):
+----+-----------+------------------------------------------------+----------+-----+
|year|    STATION|                                            NAME|      DATE|  MAX|
+----+-----------+------------------------------------------------+----------+-----+
|2015|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2015-06-12| 91.9|
|2016|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2016-07-24| 93.9|
|2017|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2017-07-22| 91.9|
|2018|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2018-07-04| 96.1|
|2019|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2019-09-30| 95.0|
|2020|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2020-07-05| 93.9|
|2021|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2021-08-12| 95.0|
|2022|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2022-06-14| 96.1|
|2023|7242979381

In [15]:
from pyspark.sql.functions import month, col

# 1) Filter to March and drop invalid MIN placeholders (9999.9)
coldest_march = (
    df2
      .filter((month(col("DATE")) == 3) & (col("MIN") != 9999.9))
      .orderBy(col("MIN").asc(), col("DATE").asc())
      .select("STATION", "NAME", "DATE", "MIN")
      .limit(1)
)

print("Coldest March day (STATION, NAME, DATE, MIN):")
coldest_march.show(truncate=False)


Coldest March day (STATION, NAME, DATE, MIN):
+-----------+------------------------------------------------+----------+---+
|STATION    |NAME                                            |DATE      |MIN|
+-----------+------------------------------------------------+----------+---+
|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2015-03-06|3.2|
+-----------+------------------------------------------------+----------+---+



In [17]:
from pyspark.sql.functions import avg, col, desc, rank
from pyspark.sql.window import Window

# 1) Exclude placeholder PRCP = 99.99 (missing)
df_prcp = df2.filter(col("PRCP") != 99.99)

# 2) Compute mean precipitation per station & year
prcp_mean = (
    df_prcp
      .groupBy("STATION", "NAME", "year")
      .agg(avg("PRCP").alias("mean_prcp"))
)

# 3) Window to pick the year with the highest mean_prcp per station
w_prcp = Window.partitionBy("STATION").orderBy(desc("mean_prcp"))

top_prcp = (
    prcp_mean
      .withColumn("rn", rank().over(w_prcp))
      .filter(col("rn") == 1)
      .select("STATION", "NAME", col("year").alias("YEAR"), col("mean_prcp"))
      .orderBy("STATION")
)

print("Year with highest average precipitation per station:")
top_prcp.show(2, truncate=False)

Year with highest average precipitation per station:
+-----------+------------------------------------------------+----+-------------------+
|STATION    |NAME                                            |YEAR|mean_prcp          |
+-----------+------------------------------------------------+----+-------------------+
|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|2018|0.15789041095890405|
|99495199999|SEBASTIAN INLET STATE PARK, FL US               |2015|0.0                |
+-----------+------------------------------------------------+----+-------------------+
only showing top 2 rows



In [19]:
from pyspark.sql.functions import count, when, col, round

# 1) Filter to 2024
df2024 = df2.filter(col("year") == 2024)

# 2) Compute missing‐GUST counts and totals per station
gust_stats = (
    df2024
      .groupBy("STATION", "NAME")
      .agg(
        count(when((col("GUST") == 999.9) | col("GUST").isNull(), True))
          .alias("missing_count"),
        count("*").alias("total_count")
      )
      # 3) Calculate percentage and round to two decimals
      .withColumn("pct_missing", round(col("missing_count") / col("total_count") * 100, 2))
      .select("STATION", "NAME", "pct_missing")
)

print("Percentage of missing GUST values in 2024:")
gust_stats.show(2, truncate=False)


Percentage of missing GUST values in 2024:
+-----------+------------------------------------------------+-----------+
|STATION    |NAME                                            |pct_missing|
+-----------+------------------------------------------------+-----------+
|72429793812|CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US|39.07      |
|99495199999|SEBASTIAN INLET STATE PARK, FL US               |100.0      |
+-----------+------------------------------------------------+-----------+



In [21]:
from pyspark.sql.functions import (
    month, col, mean as _mean, stddev as _stddev,
    expr, desc, asc, row_number, round
)
from pyspark.sql.window import Window

# 1) Filter to Cincinnati in 2020
df_cin2020 = df2.filter(
    (col("year") == 2020) &
    (col("STATION") == "72429793812")
)

# 2) Compute mean, median (50th percentile), and stddev per month
stats = (
    df_cin2020
      .groupBy(month(col("DATE")).alias("month"))
      .agg(
         _mean("TEMP").alias("mean_temp"),
         expr("percentile_approx(TEMP, 0.5)").alias("median_temp"),
         _stddev("TEMP").alias("stddev_temp")
      )
)

# 3) Compute mode per month by counting and picking the top TEMP
temp_counts = (
    df_cin2020
      .select(month(col("DATE")).alias("month"), "TEMP")
      .groupBy("month", "TEMP")
      .count()
)
w = Window.partitionBy("month").orderBy(desc("count"), asc("TEMP"))
mode_df = (
    temp_counts
      .withColumn("rn", row_number().over(w))
      .filter(col("rn") == 1)
      .select("month", col("TEMP").alias("mode_temp"))
)

# 4) Join stats + mode, round to two decimals, and show all 12 months
result = (
    stats.join(mode_df, on="month")
         .orderBy("month")
         .select(
            col("month"),
            round(col("mean_temp"), 2).alias("mean_temp"),
            round(col("median_temp"),2).alias("median_temp"),
            col("mode_temp"),
            round(col("stddev_temp"),2).alias("stddev_temp")
         )
)

print("TEMP statistics by month for Cincinnati (2020):")
result.show(12, truncate=False)


TEMP statistics by month for Cincinnati (2020):
+-----+---------+-----------+---------+-----------+
|month|mean_temp|median_temp|mode_temp|stddev_temp|
+-----+---------+-----------+---------+-----------+
|1    |37.95    |37.7       |24.7     |8.35       |
|2    |36.59    |36.0       |25.9     |7.9        |
|3    |49.07    |47.8       |39.6     |8.78       |
|4    |51.78    |51.0       |39.2     |7.31       |
|5    |60.89    |63.7       |73.9     |9.31       |
|6    |72.55    |73.7       |70.7     |4.9        |
|7    |77.6     |77.9       |72.5     |2.34       |
|8    |73.35    |73.7       |67.4     |3.49       |
|9    |66.1     |65.8       |54.7     |7.12       |
|10   |55.19    |54.0       |41.4     |6.73       |
|11   |48.0     |47.7       |47.7     |6.83       |
|12   |35.99    |35.2       |32.1     |6.64       |
+-----+---------+-----------+---------+-----------+



In [27]:
from pyspark.sql.functions import col, pow, to_date
from pyspark.sql.window import Window

# 1) Filter to Cincinnati (station=72429793812), year 2017, TEMP<50, valid MXSPD>3
df17 = (
    df_all
      .withColumn("DATE", to_date(col("DATE"), "yyyy-MM-dd"))
      .filter(
          (col("station") == "72429793812") &
          (col("year")    == 2017) &
          (col("TEMP")    <  50)   &
          (col("MXSPD")   >   3)   &  # use MXSPD for wind speed
          (col("MXSPD")   < 999.9)      # drop any placeholder codes
      )
)

# 2) Compute Wind Chill (WC)
df17_wc = df17.withColumn(
    "WC",
    35.74
    + 0.6215 * col("TEMP")
    - 35.75 * pow(col("MXSPD"), 0.16)
    + 0.4275 * col("TEMP") * pow(col("MXSPD"), 0.16)
)

# 3) Take the 10 lowest‐WC days
top10_wc = (
    df17_wc
      .orderBy(col("WC").asc())
      .select("DATE", "TEMP", "MXSPD", "WC")
      .limit(10)
)

print("Top 10 lowest Wind Chill days (Cincinnati, 2017):")
top10_wc.show(10, truncate=False)


Top 10 lowest Wind Chill days (Cincinnati, 2017):
+----------+----+-----+-------------------+
|DATE      |TEMP|MXSPD|WC                 |
+----------+----+-----+-------------------+
|2017-01-07|10.5|11.1 |-3.681340534769049 |
|2017-12-31|11.0|9.9  |-2.2286841778884705|
|2017-12-27|13.0|9.9  |0.24818110827288642|
|2017-01-06|13.6|9.9  |0.9912406941212932 |
|2017-12-28|13.6|8.9  |1.7210396860019994 |
|2017-01-08|15.9|8.9  |4.545464466491657  |
|2017-12-30|21.6|12.0 |9.70258922780936   |
|2017-12-29|21.6|9.9  |10.898701838766705 |
|2017-01-05|22.2|11.1 |10.94167046190514  |
|2017-12-25|25.8|22.0 |11.238327744614512 |
+----------+----+-----+-------------------+



In [29]:
from pyspark.sql.functions import col, count

# Count days with any extreme‐weather indicator (FRSHTT > 0) for Florida
extreme_days_df = (
    df_all
      .filter((col("station") == "99495199999") & (col("FRSHTT") > 0))
      .agg(count("*").alias("extreme_days"))
)

print("Number of extreme weather days for Florida (2015–2024):")
extreme_days_df.show()


Number of extreme weather days for Florida (2015–2024):
+------------+
|extreme_days|
+------------+
|           0|
+------------+



In [35]:
# Forecast maximum Temperature for Cincinnati (station 72429793812)
# for November and December 2024 using the last two years of data.

from pyspark.sql.functions import to_date, month, col, max as _max
import pandas as pd

# 1) Compute historical monthly maxima (excluding placeholder 9999.9)
monthly_max_valid = (
    df_all
      .withColumn("DATE", to_date(col("DATE"), "yyyy-MM-dd"))
      .filter(
          (col("station") == "72429793812") &
          (col("MAX")     != 9999.9) &
          (month(col("DATE")).isin(11, 12)) &
          (col("year").between(2022, 2023))
      )
      .withColumn("month", month(col("DATE")))
      .groupBy("year", "month")
      .agg(_max("MAX").alias("monthly_max"))
      .orderBy("year", "month")
)

print("Historical monthly maxima (2022–2023):")
monthly_max_valid.show()

# 2) Convert to Pandas for simple modeling
pdf = monthly_max_valid.toPandas()

predictions = {}

# 3A) November: linear extrapolation from two data points
nov = pdf[pdf["month"] == 11].sort_values("year")
x, y = nov["year"].values, nov["monthly_max"].values
slope = (y[1] - y[0]) / (x[1] - x[0])
intercept = y[0] - slope * x[0]
predictions["November"] = slope * 2024 + intercept

# 3B) December: only one valid point in 2023 → carry forward
dec = pdf[pdf["month"] == 12].sort_values("year")
if len(dec) == 2:
    x, y = dec["year"].values, dec["monthly_max"].values
    slope = (y[1] - y[0]) / (x[1] - x[0])
    intercept = y[0] - slope * x[0]
    predictions["December"] = slope * 2024 + intercept
else:
    predictions["December"] = dec["monthly_max"].iloc[-1]

# 4) Print forecasts
print("\nPredicted maximum temperatures for Cincinnati (2024):")
for m, val in predictions.items():
    print(f"• {m} 2024: {val:.1f} °F")


Historical monthly maxima (2022–2023):
+----+-----+-----------+
|year|month|monthly_max|
+----+-----+-----------+
|2022|   11|       75.9|
|2022|   12|       66.0|
|2023|   11|       80.1|
|2023|   12|       64.0|
+----+-----+-----------+


Predicted maximum temperatures for Cincinnati (2024):
• November 2024: 84.3 °F
• December 2024: 62.0 °F
